> **LangChain 1.x (2026)** — built on `langchain-core==1.2.30`, `langchain==1.0.0`. See `UPDATE_2026.md`.

# Chapter 6 — Molecular Identity & Standardization (v2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare/blob/main/notebooks/Chapter%2006.%20LangChain%20for%20Chemistry/LC4LSH_Chapter_6_Molecular_Identity_and_Standardization.ipynb)

**Learning objectives**
- Parse SMILES/InChI/SDF robustly with RDKit
- Handle salts/fragments and choose a parent
- Detect stereochemistry and tautomer policy issues
- Produce duplicate/conflict reports

> Runtime: ~5 min (local, no API)  
> Cost: free  
> Data: small built-in molecule set


## Environment setup


### Secrets (optional LLM only)


In [ ]:
import os
try:
    from google.colab import userdata  # type: ignore
    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False
if not IN_COLAB:
    try:
        from dotenv import load_dotenv  # type: ignore
        load_dotenv()
    except Exception:
        pass

def get_secret(name, default=None):
    if IN_COLAB and userdata is not None:
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    return os.getenv(name, default)

# These notebooks run locally on RDKit; an LLM is OPTIONAL for narrative only.
API_KEY_PROVIDER = "OPENAI"  # "GEMINI" | "OPENAI" | "GROQ" | "ANTHROPIC"
if API_KEY_PROVIDER == "OPENAI":
    os.environ["OPENAI_API_KEY"] = get_secret("LC4LS_OPENAI_API_KEY", "sk-...")
print("Optional LLM provider:", API_KEY_PROVIDER, "(RDKit runs without it)")
os.environ["HF_TOKEN"] = get_secret("HF_TOKEN", "") or ""


### Install pinned dependencies


In [ ]:
%pip install -q "rdkit==2023.9.6" "langchain==1.0.0" "langchain-core==1.2.30" "langchain-openai==1.0.0" "pandas>=2.0" "matplotlib>=3.8" "scipy>=1.11" python-dotenv
# Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)


In [ ]:
LANGSMITH_API_KEY = get_secret("LANGSMITH_API_KEY", "")
LANGSMITH_PROJECT = "lc4lsh-chapter6-mol-identity"
if LANGSMITH_API_KEY and LANGSMITH_API_KEY.startswith(("lsv2_", "ls__")):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    print("LangSmith ON ->", LANGSMITH_PROJECT)
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith OFF (fine — these notebooks are local/RDKit-first)")


## Why molecular identity matters

Before any modeling, you must know **which molecule** a record refers to. The same compound appears as many SMILES strings, salt forms, and tautomers. Standardization makes identity **reproducible and comparable**.


## 1. Parse SMILES and report problems


In [ ]:
from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

SMILES = ["CC(=O)OC1=CC=CC=C1C(=O)O", "c1ccccc1", "not_a_smiles", "C(C(C(C"]
for s in SMILES:
    m = Chem.MolFromSmiles(s)
    print(f"{s:35} -> {'OK ' + Chem.MolToSmiles(m) if m else 'PARSE FAILED'}")


## 2. Canonicalization & InChIKey as identity


In [ ]:
from rdkit.Chem import inchi

asp1 = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O")
asp2 = Chem.MolFromSmiles("O=C(O)c1ccccc1OC(=O)C")
print("canonical 1:", Chem.MolToSmiles(asp1))
print("canonical 2:", Chem.MolToSmiles(asp2))
print("same canonical?", Chem.MolToSmiles(asp1) == Chem.MolToSmiles(asp2))
print("InChIKey:", inchi.MolToInchiKey(asp1))


## 3. Salts & fragments: pick the parent


In [ ]:
from rdkit.Chem import SaltRemover
from rdkit.Chem import rdMolStandardize

salt = Chem.MolFromSmiles("[Na+].CC(=O)[O-].c1ccncc1")
remover = SaltRemover.SaltRemover()
print("stripped:", Chem.MolToSmiles(remover.StripMol(salt)))
largest = rdMolStandardize.LargestFragmentChooser().choose(salt)
print("largest fragment:", Chem.MolToSmiles(largest))


## 4. Stereochemistry & tautomer policy


In [ ]:
from rdkit.Chem import rdMolStandardize

chiral = Chem.MolFromSmiles("C[C@H](O)c1ccccc1")
Chem.AssignStereochemistry(chiral, cleanIt=True, force=True)
flags = [a.GetProp("_CIPCode") for a in chiral.GetAtoms() if a.hasProp("_CIPCode")]
print("stereocenters (CIP):", flags)

te = rdMolStandardize.TautomerEnumerator()
keto = Chem.MolFromSmiles("CC(=O)C")
enol = Chem.MolFromSmiles("CC(O)=C")
print("canonical tautomer keto:", Chem.MolToSmiles(te.Canonicalize(keto)))
print("canonical tautomer enol:", Chem.MolToSmiles(te.Canonicalize(enol)))


## 5. Duplicate & conflict report


In [ ]:
import pandas as pd

records = pd.DataFrame({
    "name": ["aspirin-a", "aspirin-b", "benzene", "asp-dup", "bad"],
    "smiles": ["CC(=O)Oc1ccccc1C(=O)O", "O=C(O)c1ccccc1OC(=O)C",
               "c1ccccc1", "CC(=O)Oc1ccccc1C(=O)O", "not_valid"]})

def canon(s):
    m = Chem.MolFromSmiles(s)
    return Chem.MolToSmiles(m) if m else None

records["canonical"] = records["smiles"].map(canon)
records["inchikey"] = records["canonical"].map(
    lambda c: inchi.MolToInchiKey(Chem.MolFromSmiles(c)) if c else None)

dup = records[records.duplicated("inchikey", keep=False) & records["inchikey"].notna()]
invalid = records[records["canonical"].isna()]
print("DUPLICATES (same InChIKey):"); print(dup[["name", "inchikey"]])
print("\nINVALID (parse failed):"); print(invalid[["name", "smiles"]])


## Limitations & safety notes

- Standardization encodes a **policy** (salt/parent/tautomer rules); document which rules you applied.
- InChIKey collapses some stereochemical/protomer distinctions depending on options.
- Identity != activity/toxicity; this notebook only establishes *which compound* a record is.
- Local/free; no API needed.


In [ ]:
# Cleanup
import gc
for _v in ("mol", "mols", "df", "llm", "model", "img", "raw", "curated"):
    globals().pop(_v, None)
try:
    import torch
    torch.cuda.empty_cache()
except Exception:
    pass
gc.collect()
print("Cleanup complete.")


## Exercises

<details><summary>Why canonicalize SMILES?</summary>Many valid SMILES describe one molecule; a canonical form gives a single string for de-duplication and joins.</details>

<details><summary>Why use InChIKey for identity?</summary>It is a fixed-length hash of the structure, robust to representation differences and easy to index.</details>

<details><summary>Why strip salts/choose a parent?</summary>Salt forms share the active parent; collapsing them avoids counting the same drug as multiple compounds.</details>

### Tasks
- **Task A** - Add a `policy` string recording which standardization steps you applied.
- **Task B** - Detect molecules whose canonical tautomer differs from the input and flag them.
- **Task C** - Count stereoisomers that share an InChIKey-connectivity layer.
- **Task D** - Export the curated table (name, canonical, InChIKey, flags) to CSV.
